In [1]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib as mpl
import json

from collections import Counter

import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_parquet("america_combined_flower_merged.parquet")

In [6]:
df = df[['Name', 'Store Name', 'flower', 'color']]
df.head()

,Name,Store Name,flower,color
0,Fake Chrysanthemum Flower in Pink Sunset Hue -...,Aflora,chrysanthemum,pink
1,Abigail Ahern Artificial Cosmos Flower Branch ...,Aflora,daisy,white
2,Bundle of 3 - Spring Palette Peony Fake Flower...,Aflora,peony,pink
3,Bundle of 3 - Spring Palette Peony Fake Flower...,Aflora,peony,yellow
4,Abigail Ahern Artificial Dahlia Flower in Beig...,Aflora,dahlia,cream


In [127]:
pd.DataFrame(df.flower.value_counts(normalize = True).head(20)).round(2)

,proportion
flower,
hydrangea,0.17
rose,0.15
orchid,0.12
peony,0.08
ranunculus,0.06
tulip,0.05
dahlia,0.03
calla lily,0.02
magnolia,0.02


#### Color for a flower

In [31]:
def flower_colors(df, flower):
    flower_df = df[df.flower == flower]
    res = pd.DataFrame(flower_df.color.value_counts(normalize=True).round(2))
    res.columns = [f'{flower}_colors']
    return res

In [135]:
flower_colors(df, 'orchid')

,orchid_colors
color,
white,0.39
pink,0.17
yellow,0.16
purple,0.08
red,0.06
green,0.05
orange,0.05
light green,0.01
magenta,0.01


In [82]:
flower_colors(df, 'hydrangea')

,hydrangea_colors
color,
white,0.27
green,0.26
blue,0.12
pink,0.11
purple,0.06
cream,0.05
red,0.03
light green,0.03
yellow,0.02


#### Flowers for a color

In [47]:
def color_flowers(df, color,  n = 10):
    color_df = df[df.color == color]
    res = pd.DataFrame(color_df.flower.value_counts(normalize=True).round(2))
    res.columns = [f'{color}_flowers']
    return res.head(n)

In [48]:
color_flowers(df, 'white')

,white_flowers
flower,
orchid,0.15
hydrangea,0.14
rose,0.14
peony,0.07
magnolia,0.05
ranunculus,0.05
tulip,0.04
calla lily,0.04
cherryblossom,0.04


In [134]:
color_flowers(df, 'red')

,red_flowers
flower,
rose,0.22
orchid,0.10
tulip,0.08
hydrangea,0.07
peony,0.07
amaryllis,0.07
ranunculus,0.05
poinsettia,0.04
dahlia,0.03


### Best Combinations

1. Get all images with multiple flowers
2. Get the value counts of each

In [113]:
def count_combinations(df: pd.DataFrame, col: str):
    """
    Count occurrences of flower type combinations in a DataFrame column,
    ignoring the order of elements in the lists.
    """
    
    # Convert lists to sorted tuples and count
    counts = df[col].apply(lambda x: tuple(sorted(x))).value_counts()
    pcts = counts / counts.sum()  # normalized
    return pd.DataFrame(pcts).round(2)

In [114]:
df = pd.read_parquet("america_combined_merged.parquet")
df = df.dropna(subset = ['plant_type', 'product_type'])
df = df.rename(columns = {'Store Name': 'store_name'})
df['how_many_flowers'] = df['flower_type'].apply(len)
df['how_many_colors'] = df['flower_colors'].apply(len)

#### Flowers

In [115]:
single_flowers = df[df.how_many_flowers == 1]
single_flower_counts =  count_combinations(single_flowers, 'flower_type')
single_flower_counts.head(20)

,count
flower_type,
"(orchid,)",0.19
"(hydrangea,)",0.14
"(rose,)",0.12
"(peony,)",0.07
"(tulip,)",0.05
"(magnolia,)",0.04
"(ranunculus,)",0.04
"(cherryblossom,)",0.03
"(calla lily,)",0.03


In [116]:
many_flowers = df[df.how_many_flowers > 1]

In [133]:
flower_combinations = count_combinations(many_flowers, 'flower_type')
flower_combinations.head(20)

,count
flower_type,
"(hydrangea, rose)",0.09
"(peony, rose)",0.03
"(hydrangea, orchid)",0.02
"(hydrangea, peony)",0.02
"(dahlia, hydrangea, rose)",0.02
"(hydrangea, ranunculus)",0.02
"(hydrangea, ranunculus, rose)",0.01
"(hydrangea, orchid, rose)",0.01
"(dahlia, rose)",0.01


#### Colors

In [118]:
single_colors = df[df.how_many_colors == 1]
single_color_counts =  count_combinations(single_colors, 'flower_colors')
single_color_counts.head(20)

,count
flower_colors,
"(white,)",0.41
"(pink,)",0.19
"(red,)",0.09
"(yellow,)",0.08
"(purple,)",0.06
"(green,)",0.04
"(orange,)",0.03
"(blue,)",0.03
"(cream,)",0.02


In [119]:
many_colors = df[df.how_many_colors > 1]
color_combos =  count_combinations(many_colors, 'flower_colors')
color_combos.head(20)

,count
flower_colors,
"(pink, white)",0.13
"(green, white)",0.07
"(green, pink)",0.03
"(red, yellow)",0.03
"(cream, pink)",0.03
"(blue, white)",0.03
"(cream, white)",0.03
"(purple, white)",0.03
"(orange, yellow)",0.03


### Combination Draft

#### Flower combination

In [73]:
def flower_complements(df, col, flower):
    
    only_this_flower_df = df[df[col] == flower]
    flower_item_names = only_this_flower_df.Name.unique().tolist()
    
    flower_items = df[df.Name.isin(flower_item_names)]
    flower_complements = flower_items[flower_items[col] != flower]

    # ratio of single vs. others
    has_other_flowers_names = flower_complements.Name.unique().tolist()
    multi_ratio =  len(has_other_flowers_names)/len(flower_item_names)
    print(f"{multi_ratio*100:.2f}% of items with {flower} have other {col}")

    # return complements
    res = pd.DataFrame(flower_complements[col].value_counts(normalize=True).round(2).head(10))
    return res
    
    
    

In [75]:
flower_complements(df, 'flower', flower)

44.95% of items with hydrangea have other flower


,proportion
flower,
rose,0.29
ranunculus,0.11
peony,0.11
dahlia,0.07
orchid,0.06
tulip,0.05
zinnia,0.03
anemone,0.02
calla lily,0.02


In [79]:
flower_complements(df, 'flower', "rose")

43.02% of items with rose have other flower


,proportion
flower,
hydrangea,0.30
peony,0.14
dahlia,0.10
ranunculus,0.09
tulip,0.06
orchid,0.04
anemone,0.03
lily,0.02
daisy,0.02


#### Color combination

In [80]:
flower_complements(df, 'color', "white")

34.24% of items with white have other color


,proportion
color,
pink,0.32
green,0.15
purple,0.11
cream,0.10
yellow,0.09
blue,0.07
peach,0.05
red,0.04
orange,0.03


In [81]:
flower_complements(df, 'color', "pink")

50.72% of items with pink have other color


,proportion
color,
white,0.31
yellow,0.13
green,0.11
purple,0.10
cream,0.09
orange,0.07
red,0.06
peach,0.06
blue,0.01
